# Factor Covariance Forecast (`fcov`) — Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building `fcov_build.ipynb`: a
forward-looking forecast of the **68×68 factor covariance matrix `F`**, the core of portfolio risk

```
σ²_p = wᵀ X F Xᵀ w  +  wᵀ Δ w
```

(`X` = exposures, `w` = weights, `Δ` = specific risk — a separate deliverable, step 07). We
replicate the USE4 covariance pipeline (Menchero, Orr & Wang 2011, §4): a layered
sequence built and tested independently.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply
untrustworthy until audited. Follow the stages linearly.

```
Layer 0  Daily CSR          daily factor returns (built upstream in daily_csr_build.ipynb)
Layer 1  EWMA base          split half-lives: vols fast, correlations slow
Layer 2  Newey-West         serial-correlation correction + daily->monthly scaling
Layer 3  Eigenfactor        Monte-Carlo eigenvalue de-biasing (the optimizer-critical step)
Layer 4  Volatility regime  scale the overall level to the current regime
Validate Bias statistics    realized/predicted ~ 1; eigenfactor bias inside the CI
```

✅ **PDF SPEC.** Factor risk is forecast from the history of factor returns; daily
returns are EWMA-weighted (volatilities react faster than correlations), corrected for
serial correlation (Newey-West) and scaled to the forecast horizon, then the eigenvalues
are de-biased by simulation, and the overall level is matched to the volatility regime.

**Why daily.** The pipeline is a daily machine: Newey-West corrects *daily* autocorrelation
and scales to the horizon, and the eigenfactor de-biasing needs many observations. The shipped
CSR is monthly (303 obs, T/N≈4.5); Layer 0 is the same constrained-WLS engine re-run **daily**
(~6,300 obs, T/N≈93) — built in `05_csr` (`daily_csr_build.ipynb`, spec
`daily_csr_spec.ipynb`) — so the rest of the pipeline is meaningful.

> **Your task.** The spec and the reference audit (`fcov_audit.md`) are provided;
> `fcov_build.ipynb` is yours to write — plus the covariance kernel module and its
> known-answer test script (§8). Finish both 05 CSR builds (`csr_build.ipynb`,
> `daily_csr_build.ipynb`) first.

## §1. Deliverables and output schema

All in `data/out/`, zstd, statistics=True.

**Inputs (built upstream — not deliverables of this step):**

- `csr_factor_returns_daily.parquet` (Layer 0) — `date`, `factor`, `factor_type`, `f`,
  `r2`, `n_stocks`. Same constrained-WLS engine as the monthly CSR, run per trading day with
  month-end exposures held fixed; built in `05_csr/daily_csr_build.ipynb`
  (spec: `daily_csr_spec.ipynb`).
- `csr_factor_returns.parquet` — the monthly CSR: its `signal_date`s are the **rebalance
  calendar** (one forecast per month-end) and its `f` values are the realized next-month
  factor returns the validation scores against.

**Factor covariance** — `fcov_factor_cov.parquet` (final forecast) and
`fcov_factor_cov_preeig.parquet` (after Newey-West + horizon, **before** eigenfactor/VRA — the
reference for the smile comparison): stacked upper triangle `est_date` (Date), `factor_i`
(String), `factor_j` (String), `cov` (Float64). One 68×68 symmetric PSD matrix per month-end
— 2,346 upper-triangle rows per `est_date` (288 forecasts / 675,648 rows per file on the
reference run).

⚠️ Only the upper triangle is stored — **consumers must mirror** to reconstruct the full
symmetric matrix. And the pre-eigen file carries neither Layer 3 nor Layer 4: the
final-vs-pre-eigen difference includes both.

**Predicted vols** — `fcov_predicted_vol.parquet`: `est_date`, `factor`, `vol`
(= √diag of the final F, monthly horizon; 19,584 rows on the reference run). The "current
forecast" is the latest `est_date`.

The covariance math must live as **pure functions** (arrays in, arrays out, no I/O) in a
kernel module of your own, validated by a known-answer test script (§8) **before** you wire
them into `fcov_build.ipynb`, which orchestrates them PIT at each rebalance date.

## §2. Pipeline & PIT discipline

```
STAGE 0  setup, parameters; load daily factor returns (Layer 0, from 05_csr)
STAGE 1  build the daily panel Φ (T×68) over the complete-factor window
STAGE 2  per month-end t (PIT, daily returns ≤ t only):
            Layer 1 EWMA base  ->  Layer 2 Newey-West + ×horizon
            ->  Layer 3 eigenfactor  ->  (Layer 4 VRA applied across dates)
STAGE 3  ship F (final + pre-eigen) and predicted vols
STAGE 4  validate: bias statistics (fcov_build.ipynb prints a summary;
            §8's validation contract is the full battery)
```

**Factor universe.** `FACTORS = ["country"] + sorted(55 industries) + sorted(12 styles)` —
assert exactly 68; country is index 0. The panel Φ starts at the first date where **all 68**
factors are present; rare later gaps (an industry absent that day) are zero-filled
(💡 master list #7). Reference run: 6,307-day panel (2001-04-02 → 2026-04-30), 288 forecasts
(2002-04-30 → 2026-03-31) after the 252-day warm-up.

**PIT.** Each `F_t` uses only daily factor returns dated ≤ `t`; the realized return it is
scored against is the next month `(t, t+1]`. The daily CSR's day-`d` return already uses the
most recent month-end exposures *strictly before* `d`, so no look-ahead enters the panel.

**Position:** after both 05 builds — `csr_build.ipynb` → `daily_csr_build.ipynb` →
**`fcov_build.ipynb`**. Step 07 (specific risk) does not consume this step's outputs (it reads
the daily CSR residuals directly); step 08 (risk decomposition) consumes `F` and runs last.

## §3. STAGE 0 — Parameters (USE4-typical; tuned against bias stats)

```python
HL_VOL      = 84     # EWMA half-life (days) for volatilities (faster)
HL_COR      = 252    # EWMA half-life (days) for correlations (slower)
NW_LAGS     = 5      # Newey-West Bartlett lags (days)
HORIZON     = 21     # forecast horizon (trading days ~ 1 month)
EIGEN_SIMS  = 1000   # eigenfactor Monte-Carlo simulations
EIGEN_A     = 1.2    # eigenfactor empirical scaling, gamma = a(v-1)+1
VRA_HL      = 42     # volatility-regime EWMA half-life (days)

# 💡 DEFAULT — implementation constants
HISTORY_CAP = 1500   # cap EWMA history per forecast (older weight negligible; speed)
MIN_HIST    = 252    # minimum daily obs before the first forecast (warm-up)
```

❓ **NOT IN PDF — exact half-lives / lags / `a`.** USE4 quotes the *structure*, not
every constant. 💡 **DEFAULT:** the values above (RiskMetrics-style split half-lives, NW
Bartlett lags, eigen scaling ≈ 1.2). All are tuned so the bias statistics sit near 1; they
are the single knobs to retune if the bias battery drifts.

💡 **DEFAULT — history cap and warm-up.** Each forecast uses at most the last
`HISTORY_CAP` = 1,500 daily rows: at the 252d half-life a day that old carries ≈ 1.6% of the
newest day's weight, and the cap also feeds Layer 3's effective-obs computation. No forecast
is produced until `MIN_HIST` = 252 daily observations exist — the first ~12 months of
rebalance dates yield no `F`.

## §4. Layer 1 — EWMA base; §5. Layer 2 — Newey-West

**EWMA base.** Weight day `t` by `2^{-(T-t)/HL}`, normalized. Compute the covariance twice —
once with `HL_VOL`, once with `HL_COR` — and recombine: volatilities from the short half-life,
correlations from the long one (`F0_ij = R_ij(HL_COR) · σ_i(HL_VOL) · σ_j(HL_VOL)`). Vol
reacts faster than correlation; this is the standard USE4/RiskMetrics split.

💡 **DEFAULT — second moments, not demeaned.** All EWMA/NW moments are computed about zero
(no demeaning): daily factor means are ≈ 0 at these horizons and estimating them adds noise.
For the lag-`l` autocovariance `Σ_t w_t f_t f_{t−l}ᵀ`, align the weights to the *later*
observation `t` (the first `l` weights drop out, without renormalizing).

**Newey-West.** Daily factor returns are serially correlated (real dynamics + nonsynchronous
trading + the cross-sectional fit). The Bartlett-weighted HAC estimator
`V = V₀ + Σ_{l=1..L}(1−l/(L+1))(V_l+V_lᵀ)` corrects it and is **PSD by construction**.
Applied to **both** the vol and corr EWMA estimates. Then **scale daily→horizon** by `×21`
— the NW correction makes that scaling unbiased w.r.t. the autocorrelation.

Per rebalance date `t` the recipe is:

```
H  = last min(HISTORY_CAP, available) daily rows of Φ up to and including t
Vv = NW(H, HL_VOL, NW_LAGS)          Vc = NW(H, HL_COR, NW_LAGS)
Fd = psd_clip(combine_vol_corr(Vv, Vc))      # daily cov — keep √diag(Fd) for Layer 4
Fh = Fd × HORIZON                            # monthly-horizon cov (the pre-eigen deliverable)
```

💡 **DEFAULT — PSD clip.** Each NW estimate is PSD, but recombining vols from one half-life
with correlations from another is **not** guaranteed PSD — symmetrize and clip negative
eigenvalues (floor ≈ 1e-16) on the combined daily matrix.

🧪 Tests to write (§8): the EWMA cov recovers a known Σ on iid data; Newey-West lifts an
AR(1) series' variance toward its long-run value.

## §6. Layer 3 — Eigenfactor risk adjustment (the optimizer-critical step)

✅ **PDF SPEC (Menchero, Wang & Orr 2011).** The sample covariance systematically
**underpredicts** the risk of its lowest-variance eigenportfolios and **overpredicts** the
highest — the bias that makes optimizers over-allocate to apparently-low-risk portfolios.

Eigendecompose `Fh = UΛUᵀ`; simulate `EIGEN_SIMS` windows of factor returns *from* `Fh` over
`T_window` days (the EWMA effective sample size); re-estimate the covariance per simulation and
measure each simulated eigenfactor's true risk (under `Fh`) against its estimated eigenvalue:

```
v(k) = √⟨ u_{sim,k}ᵀ F u_{sim,k} / λ_{sim,k} ⟩      (>1 = underestimated)
γ(k) = a·(v(k) − 1) + 1 ;   Λ_adj = Λ·γ²            (a ≈ 1.2 empirical scaling)
```

`F_eig = U·diag(Λ_adj)·Uᵀ` (re-symmetrized). The de-biasing lifts the under-predicted low
eigenfactors, so the bias-statistic "smile" flattens toward 1.

❓ **NOT IN PDF — `T_window`, `a`, equal- vs EWMA-weighted simulation.** 💡 **DEFAULT:**
`T_window` = EWMA effective obs `1/Σw²` of the (capped) history at `HL_VOL`, rounded
(≈ 242 for a full 1,500-day window); equal-weight sample cov in the simulation (the common
simplification — the bias depends mainly on `T/K`); `a ≈ 1.2`.

💡 **DEFAULT — determinism and guards.** Seed the Monte-Carlo per `est_date` (the reference
build uses the rebalance-date index), so reruns are bit-identical while each date gets
independent draws. Clip eigenvalues at ~1e-16 before the √/division — a near-singular `Fh`
must not crash, though tiny eigenvalues make the `v` ratios volatile.

🧪 Tests to write (§8): on low-`T` simulated data the bias curve has `v(small) > 1 > v(large)`
and the adjusted `F` stays PSD.

## §7. Layer 4 — Volatility regime

The model runs systematically hot or cold in regimes. Each day, the cross-sectional bias
`B_t = mean_k (f_{k,t}/σ_{k,t})²` (predicted daily vol from the most recent forecast);
`λ_vol = √EWMA(B_t)` (half-life `VRA_HL`, causal); `F_VRA = λ_vol²·F`. A uniform level
scaling — it shifts every bias statistic together, so it does **not** change the eigenfactor
smile's shape.

💡 **DEFAULT — σ source (strictly PIT).** `σ_{k,t}` is the **daily** vol `√diag(Fd)`
(pre-horizon, pre-eigenfactor, pre-VRA) from the most recent forecast dated ≤ `t`. Days
before the first forecast have no `B_t` and get `λ_vol = 1`.

💡 **DEFAULT — causal EWMA recursion.** `s ← α·B_t + (1−α)·s` with `α = 1 − 0.5^(1/VRA_HL)`,
seeded `s = B_first` — early `λ_vol` values are noisy until the 42-day EWMA burns in.

The shipped forecast is `F_t = λ_vol(t)² · F_eig,t`. Reference run: `λ_vol` mean **1.025**,
range **0.69–2.22** — it scales the model up in crises.

🧪 Tests to write (§8): `λ_vol` → 1 when the bias is identically 1 and → 2 when the bias is
identically 4.

## §8. Validation

**Bias statistics.** For each (eigen)factor, the standardized return `r_{k,t+1}/σ_{k,t}`
should have std ≈ 1 over the backtest (95% half-width ≈ `√(2/T)`). Per-factor bias uses the
final F's monthly vols against realized next-month factor returns (tolerate missing months);
the eigenfactor smile compares pre→post **in the same per-date eigenbasis** — post rescales
each eigenvalue by `γ²` — skipping months with any missing realized return.

**Kernel known-answer tests — write these before the build.** Implement the layer math as
pure functions in a module of your own and validate it with a known-answer test script
(synthetic data with known truth, fixed RNG seed, PASS/FAIL per test, non-zero exit on any
failure). The reference implementation's battery — reproduce all eight:

1. EWMA weights sum to 1 and are strictly recent-heaviest.
2. EWMA second moment with a huge half-life equals the uniform sample second moment
   (atol 1e-8), and recovers a known 4×4 Σ at T = 40,000 (relative Frobenius error < 0.05).
3. Vol/corr recombination: diagonal from the vol matrix, correlations from the corr matrix,
   exactly.
4. Newey-West on an AR(1) with ρ = 0.5 (T = 60,000, 20 lags) lifts the variance above `V₀`
   and lands within 10% of the long-run value `1/(1−ρ)²`.
5. Eigenfactor adjustment on a random 10×10 covariance (T_window = 30, 400 sims, a = 1):
   `v[smallest] > 1 > v[largest]`, and the adjusted matrix stays PSD (min eig > −1e-10).
6. VRA multiplier → 1 on bias ≡ 1 (tol 1e-6) and → 2 on bias ≡ 4 (tol 1e-3).
7. Bias statistics ≈ 1 (max |b−1| < 0.05) when the predicted vol is the true vol.
8. PSD clip of an indefinite matrix yields a PSD matrix.

**Validation contract** (run over the full forecast history):

| check | gate | reference run (288 forecasts) |
|---|---|---|
| 1 | every `F` 68×68 **symmetric + PSD** at every `est_date`: min eigenvalue > −1e-10, max asymmetry < 1e-12 | min eig 3.9×10⁻⁶, asymmetry 0 |
| 2 | mean per-factor bias ∈ [0.85, 1.15] | mean **0.949**, country 0.911, median 0.966 |
| 3 | eigenfactor smile flattens: mean \|b−1\| post < pre **and** low-10-eigenfactor mean \|b−1\| post < pre | **0.310 → 0.074** overall; low-10 0.61 → 0.11; 97% of eigenfactors within 2·CI of 1 |
| 4 | sane predicted vols: median country monthly vol ∈ [2, 8]% and every diagonal > 0 | median 4.14% |
| 5 | OOS split-half robustness: each half of the forecast history independently passes checks 2 and 3 | early: bias 0.941, smile 0.31 → 0.10; late: bias 0.949, 0.31 → 0.06 |

Nothing is fit: the half-lives and `a = 1.2` are fixed priors and every forecast is strictly
PIT, so check 5 is honest out-of-sample evidence. Pre-adjustment, the lowest-variance
eigenfactors realize up to ≈ **1.95×** their predicted risk — check 3 is the headline result
this stage exists for.

Plus informational prints: VRA multiplier stats (mean/min/max) and a latest-forecast snapshot
— country monthly vol (~4–5% on the reference run), PSD min eigenvalue, top-eigenfactor
loadings. Layer-0 quality (daily↔monthly reconciliation, condition numbers) is gated upstream
in `05_csr/daily_csr_spec.ipynb`, not here.

## §9. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Where to revisit |
|---|---|---|---|
| 1 | Daily vs monthly returns | Daily CSR (Layer 0) — USE4 is a daily machine | Never — monthly guts Newey-West and the eigenfactor de-biasing |
| 2 | EWMA half-lives | 84d vol / 252d corr (split; vol faster) | Retune against the per-factor bias battery |
| 3 | Newey-West lags | 5 daily Bartlett lags | If factor-return autocorrelation profile changes |
| 4 | Horizon scaling | ×21 daily after the NW correction | If the forecast horizon changes |
| 5 | Eigenfactor `T_window` / `a` | EWMA effective obs; `a≈1.2`; equal-weight sim | Tune `a` so eigenfactor bias sits inside the CI |
| 6 | VRA half-life | 42d causal EWMA of the cross-sectional bias | If regime tracking lags or over-reacts |
| 7 | Complete-panel window | start at the first all-68-present day; fill rare later gaps with 0 | If an industry goes persistently absent |
| 8 | Storage | stacked upper-triangle per est_date; keep pre-eigen F for the smile | If read patterns demand it — pack to a dense 3-D array |
| 9 | History cap | last 1,500 days per forecast (≈ 1.6% residual weight at the 252d half-life) | If half-lives lengthen materially |
| 10 | Warm-up | first forecast only after 252 daily obs (~12 months of rebalance dates skipped) | Never — earlier forecasts are undersampled |

**Out of scope (documented alternatives):** GARCH-family (DCC for time-varying correlations,
GJR/EGARCH for vol asymmetry) and Ledoit-Wolf shrinkage — at 68×68 the factor model is already
the dimension reduction, so the eigenfactor adjustment plays the bias-correction role shrinkage
would at the asset level. Specific risk `Δ` (the diagonal term) is a separate deliverable
(step 07).